# Phase 7 — Data Quality Framework

This notebook implements automated data-quality validation for:

`genai_copilot.silver.sales`

Quality results are stored in:

`genai_copilot.gold.data_quality_results`

The framework validates:

- completeness
- uniqueness
- validity
- consistency
- business rules

The notebook:

1. Loads Silver data
2. Runs 10 quality checks
3. Creates a quality-results DataFrame
4. Calculates the overall quality status
5. Persists results to Gold
6. Applies a data-quality gate

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    TimestampType,
)

from datetime import datetime
import uuid

In [0]:
SILVER_TABLE = "genai_copilot.silver.sales"
DQ_TABLE = "genai_copilot.gold.data_quality_results"

print("Source table:", SILVER_TABLE)
print("DQ results table:", DQ_TABLE)

In [0]:
df = spark.table(SILVER_TABLE)

total_records = df.count()

print(f"Total Silver records: {total_records}")

display(df.limit(10))

In [0]:
run_id = str(uuid.uuid4())
run_timestamp = datetime.now()

results = []

print("Run ID:", run_id)
print("Run timestamp:", run_timestamp)

In [0]:
def add_result(
    check_id,
    check_name,
    check_type,
    failed_records,
    message
):
    status = "PASS" if failed_records == 0 else "FAIL"

    failure_rate = (
        failed_records / total_records
        if total_records > 0
        else 0.0
    )

    results.append({
        "run_id": run_id,
        "check_id": check_id,
        "table_name": SILVER_TABLE,
        "check_name": check_name,
        "check_type": check_type,
        "status": status,
        "failed_records": int(failed_records),
        "total_records": int(total_records),
        "failure_rate": float(failure_rate),
        "message": message,
        "run_timestamp": run_timestamp,
    })

In [0]:
failed = df.filter(
    F.col("order_id").isNull()
).count()

add_result(
    "DQ001",
    "order_id_not_null",
    "completeness",
    failed,
    "order_id must not be NULL"
)

print(
    f"DQ001: order_id_not_null - "
    f"{'PASS' if failed == 0 else 'FAIL'} "
    f"({failed} failed)"
)

In [0]:
duplicate_order_ids = (
    df
    .filter(F.col("order_id").isNotNull())
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_result(
    "DQ002",
    "order_id_unique",
    "uniqueness",
    duplicate_order_ids,
    "order_id should uniquely identify an order"
)

print(
    f"DQ002: order_id_unique - "
    f"{'PASS' if duplicate_order_ids == 0 else 'FAIL'} "
    f"({duplicate_order_ids} failed)"
)

In [0]:
failed = df.filter(
    F.col("order_date").isNull()
).count()

add_result(
    "DQ003",
    "order_date_not_null",
    "completeness",
    failed,
    "order_date must not be NULL"
)

print(
    f"DQ003: order_date_not_null - "
    f"{'PASS' if failed == 0 else 'FAIL'} "
    f"({failed} failed)"
)

In [0]:
failed = df.filter(
    F.col("quantity").isNull()
    | (F.col("quantity") <= 0)
).count()

add_result(
    "DQ004",
    "quantity_positive",
    "validity",
    failed,
    "quantity must be greater than zero"
)

print(
    f"DQ004: quantity_positive - "
    f"{'PASS' if failed == 0 else 'FAIL'} "
    f"({failed} failed)"
)

In [0]:
failed = df.filter(
    F.col("discount").isNull()
    | (F.col("discount") < 0)
    | (F.col("discount") > 1)
).count()

add_result(
    "DQ005",
    "discount_valid_range",
    "validity",
    failed,
    "discount must be between 0 and 1"
)

print(
    f"DQ005: discount_valid_range - "
    f"{'PASS' if failed == 0 else 'FAIL'} "
    f"({failed} failed)"
)

In [0]:
failed = df.filter(
    F.col("revenue").isNull()
    | (F.col("revenue") < 0)
).count()

add_result(
    "DQ006",
    "revenue_non_negative",
    "validity",
    failed,
    "revenue must be greater than or equal to zero"
)

print(
    f"DQ006: revenue_non_negative - "
    f"{'PASS' if failed == 0 else 'FAIL'} "
    f"({failed} failed)"
)

In [0]:
failed = df.filter(
    F.col("cost").isNull()
    | (F.col("cost") < 0)
).count()

add_result(
    "DQ007",
    "cost_non_negative",
    "validity",
    failed,
    "cost must be greater than or equal to zero"
)

print(
    f"DQ007: cost_non_negative - "
    f"{'PASS' if failed == 0 else 'FAIL'} "
    f"({failed} failed)"
)

In [0]:
profit_check = (
    df
    .withColumn(
        "expected_profit",
        F.col("revenue") - F.col("cost")
    )
    .withColumn(
        "profit_difference",
        F.abs(
            F.col("profit") - F.col("expected_profit")
        )
    )
)

failed = profit_check.filter(
    F.col("profit_difference") > 0.01
).count()

add_result(
    "DQ008",
    "profit_calculation_consistent",
    "consistency",
    failed,
    "profit should equal revenue minus cost within 0.01 tolerance"
)

print(
    f"DQ008: profit_calculation_consistent - "
    f"{'PASS' if failed == 0 else 'FAIL'} "
    f"({failed} failed)"
)

In [0]:
valid_statuses = [
    "Completed",
    "Cancelled",
    "Returned",
    "Pending",
]

failed = df.filter(
    F.col("order_status").isNull()
    | ~F.col("order_status").isin(valid_statuses)
).count()

add_result(
    "DQ009",
    "order_status_valid",
    "validity",
    failed,
    "order_status must belong to the approved status list"
)

print(
    f"DQ009: order_status_valid - "
    f"{'PASS' if failed == 0 else 'FAIL'} "
    f"({failed} failed)"
)

In [0]:
valid_channels = [
    "Online",
    "Retail Store",
    "Partner",
]

failed = df.filter(
    F.col("sales_channel").isNull()
    | ~F.col("sales_channel").isin(valid_channels)
).count()

add_result(
    "DQ010",
    "sales_channel_valid",
    "validity",
    failed,
    "sales_channel must belong to the approved channel list"
)

print(
    f"DQ010: sales_channel_valid - "
    f"{'PASS' if failed == 0 else 'FAIL'} "
    f"({failed} failed)"
)

In [0]:
result_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("check_id", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("check_name", StringType(), False),
    StructField("check_type", StringType(), False),
    StructField("status", StringType(), False),
    StructField("failed_records", LongType(), False),
    StructField("total_records", LongType(), False),
    StructField("failure_rate", DoubleType(), False),
    StructField("message", StringType(), False),
    StructField("run_timestamp", TimestampType(), False),
])

results_df = spark.createDataFrame(
    results,
    schema=result_schema
)

display(
    results_df
    .select(
        "check_id",
        "check_name",
        "status",
        "failed_records",
        "failure_rate",
        "message"
    )
    .orderBy("check_id")
)

In [0]:
failed_checks = sum(
    1
    for result in results
    if result["status"] == "FAIL"
)

total_checks = len(results)

overall_status = (
    "PASS"
    if failed_checks == 0
    else "FAIL"
)

print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)
print(f"Total checks: {total_checks}")
print(f"Failed checks: {failed_checks}")
print(f"Overall status: {overall_status}")
print("=" * 60)

In [0]:
failed_df = (
    results_df
    .filter(F.col("status") == "FAIL")
    .select(
        "check_id",
        "check_name",
        "failed_records",
        "total_records",
        "failure_rate",
        "message"
    )
)

if failed_checks == 0:
    print("No failed data-quality checks.")
else:
    display(failed_df)

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS genai_copilot.gold
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS genai_copilot.gold.data_quality_results (
    run_id STRING,
    check_id STRING,
    table_name STRING,
    check_name STRING,
    check_type STRING,
    status STRING,
    failed_records BIGINT,
    total_records BIGINT,
    failure_rate DOUBLE,
    message STRING,
    run_timestamp TIMESTAMP
)
USING DELTA
""")

print(
    "Verified:",
    DQ_TABLE
)

In [0]:
(
    results_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(DQ_TABLE)
)

print(
    f"Successfully persisted {len(results)} "
    f"quality checks."
)

In [0]:
display(
    spark.table(DQ_TABLE)
    .filter(F.col("run_id") == run_id)
    .select(
        "check_id",
        "check_name",
        "status",
        "failed_records",
        "failure_rate",
        "run_timestamp"
    )
    .orderBy("check_id")
)

In [0]:
if overall_status == "FAIL":

    failed_ids = [
        result["check_id"]
        for result in results
        if result["status"] == "FAIL"
    ]

    raise ValueError(
        "DATA QUALITY GATE FAILED. "
        f"Failed checks: {failed_ids}"
    )

print("DATA QUALITY GATE: PASS")
print("Silver data is ready for downstream processing.")

In [0]:
%sql
SELECT
    check_id,
    check_name,
    status,
    failed_records
FROM genai_copilot.gold.data_quality_results
WHERE run_id = (
    SELECT MAX(run_id)
    FROM genai_copilot.gold.data_quality_results
)
ORDER BY check_id;

In [0]:
%sql
WITH latest AS (
    SELECT MAX(run_timestamp) AS latest_timestamp
    FROM genai_copilot.gold.data_quality_results
)
SELECT
    check_id,
    check_name,
    status,
    failed_records,
    total_records
FROM genai_copilot.gold.data_quality_results
WHERE run_timestamp = (
    SELECT latest_timestamp
    FROM latest
)
ORDER BY check_id;